# Experiment 4: linear track with an enriched 4D latent state

**Research question.** How do velocity and context affect recovery of the track state?

This is a transparent walkthrough. Every scientific stage is a separate cell;
there is no call to the all-in-one experiment runner.

## Pipeline map

| Stage | Module used | Main output |
|---|---|---|
| Task latent | `neurobridge.data.sim.LatentTrajectoryGenerator` | `Z`, condition, state |
| Population map | `build_structured_B` or linear place-field builder | `B`, neuron types |
| Spike emission | `drive_to_rate`, `rate_to_spike` | `u`, `lam`, `X` |
| Windows | `build_windows_and_labels` | `TemporalWindowDataset`, metadata |
| Split | `split_trials` | train/test trials and masks |
| Linear baseline | `sklearn.decomposition.PCA` | PCA embedding |
| Neural encoder | `TemporalCNNEncoder` | CNN embedding |
| Target and loss | `build_similarity_matrix`, `soft_contrastive_loss` | batch target `Q`, scalar loss |
| Evaluation | `evaluate_models` | held-out recovery metrics |

To modify one stage, edit its cell and rerun that cell plus the cells below it.

## 0. Locate the repository

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src" / "neurobridge").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src" / "neurobridge").exists():
    raise FileNotFoundError("Open this notebook from the Neuro_Bridge repository.")

sys.path.insert(0, str(PROJECT_ROOT / "src"))
print("Project root:", PROJECT_ROOT)

## 1. Import the modules

These imports expose the complete dependency path. No experiment-wide wrapper
is imported.

In [ ]:
from dataclasses import asdict
import json
import random

import joblib
import numpy as np
import torch
from sklearn.decomposition import PCA
from torch.utils.data import DataLoader, Subset

from neurobridge.data.sim import LatentTrajectoryGenerator
from neurobridge.experiments import (
    SyntheticTaskConfig,
    build_linear_loading_and_place_fields,
    build_similarity_matrix,
    build_windows_and_labels,
    evaluate_models,
    save_experiment_figures,
    split_trials,
)

from neurobridge.data.sim.builders import drive_to_rate, rate_to_spike
from neurobridge.losses.infonce import soft_contrastive_loss
from neurobridge.models.temporal_cnn import TemporalCNNEncoder
from neurobridge.train.loop import encode_windows, train_epoch

## 2. Set the experimental parameters

This is the main control panel. Change trial count, latent dimension, neuron
count, window size, loss weights, temperatures, epochs, or seed here.

In [ ]:
CONFIG = SyntheticTaskConfig(
    name="experiment_04_linear_4d",
    condition_mode="linear",
    latent_dim=4,
    n_trials=160,
    trial_length=100,
    n_neurons=100,
    window_size=10,
    stride=1,
    place_fraction=0.25,
    place_width=0.10,
    place_scale=3.0,
    cnn_epochs=30,
    batch_size=256,
    learning_rate=1e-3,
    temperature=0.1,
    similarity_tau=0.5,
    time_weight=0.5,
    label_weight=0.5,
    dt=0.02,
    random_state=42,
)
CONFIG

## 3. Fix randomness and create the output directory

Three random-number generators are seeded because NumPy, Python, and PyTorch
are all used by different stages.

In [ ]:
random.seed(CONFIG.random_state)
np.random.seed(CONFIG.random_state)
torch.manual_seed(CONFIG.random_state)

OUTPUT_ROOT = PROJECT_ROOT / "outputs" / CONFIG.name
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("output:", OUTPUT_ROOT)

## 4. Generate the known latent process

`Z` has shape `(trials, time bins, latent dimensions)`. This is the ground
truth that the encoders will later attempt to recover from spike counts.

In [ ]:
conditions = (
    np.arange(CONFIG.n_conditions)
    if CONFIG.condition_mode == "circular"
    else None
)

latent_generator = LatentTrajectoryGenerator(
    n_trials=CONFIG.n_trials,
    L=CONFIG.trial_length,
    k=CONFIG.latent_dim,
    phi=CONFIG.phi,
    conditions=conditions,
    condition_mode=CONFIG.condition_mode,
    n_conditions=CONFIG.n_conditions,
    noise_scale=CONFIG.noise_scale,
    condition_type="balanced",
)

Z, condition, state = latent_generator.generate_latent(return_state=True)

print("Z shape:", Z.shape)
print("condition shape:", None if condition is None else condition.shape)
print("state variables:", sorted(state))
print("first latent state:", Z[0, 0])

## 5. Map the latent state into a neural population

The mapping retains position-, direction-, mixed-, and place-selective units and adds velocity- and context-selective units.

`B` has shape `(latent dimensions, neurons)`. Entry `B[k, n]` controls how
latent coordinate `k` contributes to neuron `n`.

In [ ]:
(
    B,
    neuron_types,
    place_centers,
    place_drive,
) = build_linear_loading_and_place_fields(
    k=CONFIG.latent_dim,
    n_neurons=CONFIG.n_neurons,
    position=state["position"][:, :, 0],
    place_fraction=CONFIG.place_fraction,
    place_width=CONFIG.place_width,
    place_scale=CONFIG.place_scale,
    first_coordinates_multiplier=CONFIG.first_coordinates_multiplier,
    random_state=CONFIG.random_state + 1,
)

unique_types, counts = np.unique(neuron_types, return_counts=True)
print("B shape:", B.shape)
print("place drive shape:", place_drive.shape)
print("neuron types:", dict(zip(unique_types, counts)))
print("place-selective neurons:", np.sum(neuron_types == "place"))

## 6. Emit stochastic spike counts

The visible computation is:

`u = Z @ B + baseline + place_drive`

`lam = rate_scale * softplus(u)`

`X ~ count_process(lam * dt)`

`u` is unconstrained neural drive, `lam` is a positive rate, and `X` is the
observed binned population activity.

In [ ]:
rng = np.random.default_rng(CONFIG.random_state + 2)
baseline = rng.normal(
    CONFIG.baseline_mean,
    CONFIG.baseline_std,
    size=CONFIG.n_neurons,
)

# Linear predictor: latent contribution + neuron baseline + nonlinear place field.
u = Z @ B + baseline + place_drive

# Positive conditional rate and stochastic binned counts.
lam = CONFIG.rate_scale * drive_to_rate(u, "softplus")
X = rate_to_spike(lam, CONFIG.dt)

print("u:", u.shape, "range", (u.min(), u.max()))
print("lambda:", lam.shape, "mean", lam.mean())
print("X:", X.shape, "mean count/bin", X.mean())
print("fraction of zero counts:", np.mean(X == 0))

## 7. Convert each trial into centered temporal windows

One observation for the encoder is a matrix with shape
`(window_size, neurons)`. Centered padding keeps one window for every original
time bin and no window crosses a trial boundary.

In [ ]:
dataset, metadata = build_windows_and_labels(
    X,
    condition,
    state,
    CONFIG,
)

sample = dataset[0]
print("number of windows:", len(dataset))
print("one CNN input x:", sample["x"].shape, "= time bins x neurons")
print("sample keys:", sorted(sample))
print("time range:", (metadata["time_id"].min(), metadata["time_id"].max()))
print("trials represented:", len(np.unique(metadata["trial_id"])))

## 8. Split complete trials

The split is made on trial identifiers, not on individual windows. Therefore
no window from a test trial is used to fit PCA or CNN1D.

In [ ]:
(
    train_trials,
    test_trials,
    train_mask,
    test_mask,
) = split_trials(metadata, condition, CONFIG)

train_indices = np.flatnonzero(train_mask)
test_indices = np.flatnonzero(test_mask)

print("train trials:", len(train_trials), train_trials[:10])
print("test trials:", len(test_trials), test_trials[:10])
print("train windows:", len(train_indices))
print("test windows:", len(test_indices))
assert not np.intersect1d(train_trials, test_trials).size

## 9. Fit PCA directly

PCA requires a 2D matrix, so each `(time, neuron)` window is flattened. PCA is
fit only on training windows and then transforms all windows in their original
order.

In [ ]:
X_windows = dataset.X_windows.numpy()
flattened_windows = X_windows.reshape(len(dataset), -1)
pca_dim = min(CONFIG.latent_dim, flattened_windows.shape[1])

pca_model = PCA(
    n_components=pca_dim,
    random_state=CONFIG.random_state,
)
pca_model.fit(flattened_windows[train_indices])
pca_embedding = pca_model.transform(flattened_windows)

print("window tensor:", X_windows.shape)
print("flattened PCA input:", flattened_windows.shape)
print("PCA embedding:", pca_embedding.shape)
print("explained variance ratio:", pca_model.explained_variance_ratio_)

## 10. Inspect the soft target for one mini-batch

The shuffled mini-batch does not retain chronological row order, but every
window carries its metadata. `build_similarity_matrix` reconstructs pairwise
temporal and task distances inside the batch and converts them into the soft
target matrix `Q`.

In [ ]:
preview_loader = DataLoader(
    Subset(dataset, train_indices.tolist()),
    batch_size=min(32, CONFIG.batch_size),
    shuffle=True,
)
preview_batch = next(iter(preview_loader))
Q_preview = build_similarity_matrix(preview_batch, CONFIG)

print("batch x:", preview_batch["x"].shape)
print("batch time_id:", preview_batch["time_id"].shape)
print("batch label:", preview_batch["label"].shape)
print("soft target Q:", Q_preview.shape)
print("Q min/mean/max:", Q_preview.min().item(), Q_preview.mean().item(), Q_preview.max().item())

## 11. Construct CNN1D, optimizer, data loader, and objectives

PyTorch receives `x` as `(batch, time, neurons)`. `TemporalCNNEncoder`
internally transposes it for `Conv1d`, which expects
`(batch, channels=neurons, sequence=time)`.

The model returns one normalized embedding vector per window. The loss compares
the model distribution induced by embedding cosine similarities with the soft
target distribution `Q`.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

cnn_model = TemporalCNNEncoder(
    n_features=CONFIG.n_neurons,
    embedding_dim=CONFIG.embedding_dim,
    hidden_dim=64,
    kernel_size=3,
    n_layers=3,
    normalize=True,
).to(device)

optimizer = torch.optim.AdamW(
    cnn_model.parameters(),
    lr=CONFIG.learning_rate,
    weight_decay=1e-4,
)

training_loader = DataLoader(
    Subset(dataset, train_indices.tolist()),
    batch_size=CONFIG.batch_size,
    shuffle=True,
    drop_last=True,
)

def loss_function(embedding, target_similarity):
    return soft_contrastive_loss(
        embedding,
        target_similarity,
        temperature=CONFIG.temperature,
    )

def similarity_function(batch):
    return build_similarity_matrix(batch, CONFIG)

print(cnn_model)
print("device:", device)
print("batches per epoch:", len(training_loader))

## 12. Train explicitly, one epoch at a time

`train_epoch` performs the repeated PyTorch mechanics for one epoch. The loop,
epoch count, model, optimizer, target builder, and loss remain visible and can
be replaced independently.

In [ ]:
cnn_losses = []

for epoch in range(CONFIG.cnn_epochs):
    epoch_loss = train_epoch(
        cnn_model,
        training_loader,
        optimizer,
        loss_function,
        device=device,
        similarity_builder=similarity_function,
    )
    cnn_losses.append(epoch_loss)

    if epoch == 0 or (epoch + 1) % 5 == 0:
        print(f"epoch {epoch + 1:02d}/{CONFIG.cnn_epochs}: loss={epoch_loss:.4f}")

## 13. Encode every window without shuffling

Training uses shuffled batches; final encoding does not. `shuffle=False`
restores exact alignment between embedding rows and `trial_id`, `time_id`, and
labels.

In [ ]:
ordered_loader = DataLoader(
    dataset,
    batch_size=CONFIG.batch_size,
    shuffle=False,
)
cnn_embedding_tensor, encoded_metadata = encode_windows(
    cnn_model,
    ordered_loader,
    device=device,
)
cnn_embedding = cnn_embedding_tensor.numpy()
cnn_losses = np.asarray(cnn_losses)

print("CNN embedding:", cnn_embedding.shape)
print("expected rows:", CONFIG.n_trials * CONFIG.trial_length)
print("final loss:", cnn_losses[-1])

## 14. Evaluate held-out latent recovery

RSA Spearman compares pairwise-distance rankings in the known latent and in the
embedding. Procrustes R^2 compares coordinates after centering, rotation,
reflection, and global scale. Neither metric should replace inspection of the
trajectory plots.

In [ ]:
fitted = {
    "pca_model": pca_model,
    "pca_embedding": pca_embedding,
    "pca_explained_variance_ratio": pca_model.explained_variance_ratio_,
    "cnn_model": cnn_model,
    "cnn_embedding": cnn_embedding,
    "cnn_losses": cnn_losses,
    "cnn_training_steps": CONFIG.cnn_epochs * len(training_loader),
}

metrics = evaluate_models(
    Z=Z,
    metadata=metadata,
    fitted=fitted,
    train_mask=train_mask,
    test_mask=test_mask,
    config=CONFIG,
)

for model_name, values in metrics.items():
    print(f"\n{model_name.upper()}")
    print(f"  test Procrustes R^2:    {values['test_procrustes_r2']:.3f}")
    print(f"  test RSA Spearman:      {values['test_rsa_spearman']:.3f}")
    print(f"  motor-core RSA:         {values['core_test_rsa_spearman']:.3f}")
    print(f"  trajectory RSA:         {values['trajectory_rsa_spearman']:.3f}")

## 15. Scientific interpretation

Compare complete-state metrics with the position-direction motor core. The comparison identifies whether added coordinates are recovered or whether the encoder mainly preserves the track cycle.

## 16. Assemble and save reproducible artifacts

This cell is deliberately explicit: it shows exactly which arrays, metadata,
models, metrics, and figures are written to disk.

In [ ]:
results = {
    "config": asdict(CONFIG),
    "device": str(device),
    "Z": Z,
    "condition": condition,
    "state": state,
    "B": B,
    "baseline": baseline,
    "neuron_types": neuron_types,
    "place_centers": place_centers,
    "u": u,
    "lam": lam,
    "X": X,
    "metadata": metadata,
    "train_trials": train_trials,
    "test_trials": test_trials,
    "metrics": metrics,
    **fitted,
}

model_root = OUTPUT_ROOT / "models"
model_root.mkdir(parents=True, exist_ok=True)
joblib.dump(pca_model, model_root / "pca.joblib")
torch.save(cnn_model.state_dict(), model_root / "cnn1d_state_dict.pt")

serializable_results = {
    key: value
    for key, value in results.items()
    if key not in {"pca_model", "cnn_model"}
}
joblib.dump(serializable_results, OUTPUT_ROOT / "results.joblib")
(OUTPUT_ROOT / "metrics.json").write_text(
    json.dumps(metrics, indent=2),
    encoding="utf-8",
)
save_experiment_figures(results, OUTPUT_ROOT)

print("saved to:", OUTPUT_ROOT)
for path in sorted(OUTPUT_ROOT.rglob("*")):
    if path.is_file():
        print(" -", path.relative_to(PROJECT_ROOT))

## Modification guide

- Change the latent process: edit Sections 2 and 4.
- Change neural tuning or place fields: edit Section 5.
- Change the observation model: edit Section 6.
- Change window size or stride: edit Section 2, then rerun from Section 7.
- Change the target geometry: edit `time_weight`, `label_weight`, or
  `similarity_tau`, or replace `similarity_function` in Section 11.
- Change the loss: replace `loss_function` in Section 11.
- Change the CNN architecture: edit the `TemporalCNNEncoder` constructor.
- Add a new model: reproduce the PCA or CNN path and add its embedding to
  `fitted` before evaluation.